# Task 4 - Game Title Detection from Screenshots (5 classes, metric: F1 macro)

CPE342 Karena Player Intelligence System

> **Rules followed:** image classification may use pre-trained weights (ImageNet-pretrained ConvNeXt-Small fine-tuned on the task images, no AutoML); every statistic (class weights, fold split, best-epoch choice) is computed from the **train** set only; the test set is used only for the final prediction.
>
> Needs `torch`, `torchvision`, `timm`, `pillow`; a GPU is strongly recommended (`QUICK=1` runs a tiny smoke test).

## 1. Import Libraries & Configuration

In [ ]:
import os, random, time, warnings, joblib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
import torchvision.transforms as T
import timm
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_class_weight
warnings.filterwarnings("ignore")

SEED = 42
QUICK = os.environ.get("QUICK") == "1"   # QUICK=1 -> small smoke test (results not meaningful)
ROOT = Path.cwd()                         # repo root, or the task4/ folder itself
OUT = ROOT / ("outputs_quick" if QUICK else "outputs")
(OUT / "models").mkdir(parents=True, exist_ok=True)
print("QUICK mode:", QUICK)

TASK = 4
NAMES = ["7-11 Knights", "FiveN", "Free Fried", "PUBE", "Roblock"]
DATA = ROOT / "task4" if (ROOT / "task4" / "train.csv").exists() else ROOT   # run from repo root or from inside task4/
MODEL_LR = {"convnext_small.fb_in22k_ft_in1k": 2e-4, "convnext_base.fb_in22k_ft_in1k": 1.5e-4, "tf_efficientnetv2_s.in21k_ft_in1k": 6e-4}
MODELS = ["convnext_small.fb_in22k_ft_in1k"] if QUICK else os.environ.get(
    "MODELS", "convnext_small.fb_in22k_ft_in1k,tf_efficientnetv2_s.in21k_ft_in1k").split(",")   # different families -> diverse errors
MODEL_NAME = "+".join(MODELS)

CFG = {
    "n_folds": 5,
    "train_folds": [0] if QUICK else [int(f) for f in os.environ.get("FOLDS", "0,1,2,3,4").split(",")],   # all 5 folds -> OOF + 5-model ensemble
    "img_size": tuple(int(v) for v in os.environ.get("IMG", "192,384").split(",")),   # landscape aspect of 230x120 screenshots; override e.g. IMG=128,256
    "batch_size": 32 if QUICK else 64,
    "epochs": 1 if QUICK else int(os.environ.get("EPOCHS", 12)),
    "head_lr_mult": 5.0,                   # classifier head learns faster than the pretrained backbone
    "weight_decay": 1e-2,
    "pseudo": os.environ.get("PSEUDO") == "1",   # OFF by default: pseudo-labelling trains on TEST images (README: stats from train only)
    "pseudo_thr": 0.98,
    "aug": os.environ.get("AUG", "strong"),      # "strong" (default): random-resized-crop + TrivialAugment + blur/grayscale -> robust to the train/test shift; "light" = previous setup
    "prior_em": os.environ.get("PRIOR_EM") == "1",   # OFF by default: label-shift correction re-estimates class shares on unlabeled TEST predictions (confirm with TA)
    "ema_decay": 0.998,                    # EMA of weights (evaluated + used for test prediction)
    "warmup_epochs": 1,
    "num_classes": 5,
    "num_workers": 0 if QUICK else 2,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
}
print("device:", CFG["device"])

def seed_everything(seed=SEED):
    random.seed(seed); os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
seed_everything()

In [ ]:
# ---------- helper functions (all statistics are fitted on TRAIN only) ----------
NORM = dict(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
light_transform = T.Compose([
    T.Resize(CFG["img_size"]),
    T.RandomHorizontalFlip(p=0.5),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    T.RandomAffine(degrees=5, translate=(0.05, 0.05), scale=(0.9, 1.05)),
    T.ToTensor(), T.Normalize(**NORM),
    T.RandomErasing(p=0.25, scale=(0.02, 0.15)),
])
# test screenshots come from other sources than train (train-vs-test classifier on thumbnails: AUC 0.76), so the model must not
# rely on exact framing / colour: random crops of 60-100 % of the image, TrivialAugment (colour, contrast, sharpness, posterize...),
# occasional blur / grayscale, random erasing of HUD parts
strong_transform = T.Compose([
    T.RandomResizedCrop(CFG["img_size"], scale=(0.6, 1.0), ratio=(1.6, 2.3), antialias=True),
    T.RandomHorizontalFlip(p=0.5),
    T.TrivialAugmentWide(),
    T.RandomGrayscale(p=0.05),
    T.RandomApply([T.GaussianBlur(5, sigma=(0.1, 1.5))], p=0.2),
    T.ToTensor(), T.Normalize(**NORM),
    T.RandomErasing(p=0.3, scale=(0.02, 0.15)),
])
train_transform = strong_transform if CFG["aug"] == "strong" else light_transform
val_transform = T.Compose([T.Resize(CFG["img_size"]), T.ToTensor(), T.Normalize(**NORM)])

def img_filenames(df):
    """Use `file_name` column from CSV when present, falling back to `<id>.jpg`."""
    if "file_name" in df.columns:
        return df["file_name"].astype(str).values
    return np.array([f"{i}.jpg" for i in df["id"].astype(str).values])

class GameDataset(Dataset):
    """Screenshot dataset: files are `<img_dir>/<file_name>` (or `<id>.jpg`); labels are optional (test)."""
    def __init__(self, df, img_dir, transform=None, target=None):
        self.files, self.img_dir, self.transform = img_filenames(df), img_dir, transform
        self.y = df[target].astype(int).values if target else None
    def __len__(self):
        return len(self.files)
    def __getitem__(self, i):
        img = Image.open(self.img_dir / self.files[i]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return (img, int(self.y[i])) if self.y is not None else (img, 0)

def make_loader(df, img_dir, transform, shuffle, target=None):
    return DataLoader(GameDataset(df, img_dir, transform, target), batch_size=CFG["batch_size"] * (1 if shuffle else 2),
                      shuffle=shuffle, num_workers=CFG["num_workers"], pin_memory=CFG["device"] == "cuda", drop_last=shuffle)

@torch.no_grad()
def predict_proba(model, loader, tta=True, scales=(1.0, 1.15)):
    """Softmax probabilities; TTA = mean over {image, left-right flip} x {native, 1.15x resolution}."""
    model.eval(); out = []
    for x, _ in loader:
        x = x.to(CFG["device"])
        with torch.autocast(CFG["device"], enabled=CFG["device"] == "cuda"):
            p = torch.softmax(model(x), 1)
            if tta:
                ps = [p, torch.softmax(model(torch.flip(x, dims=[3])), 1)]
                for s in scales[1:]:
                    xs = nn.functional.interpolate(x, scale_factor=s, mode="bilinear", align_corners=False)
                    ps += [torch.softmax(model(xs), 1), torch.softmax(model(torch.flip(xs, dims=[3])), 1)]
                p = torch.stack(ps).mean(0)
        out.append(p.float().cpu().numpy())
    return np.concatenate(out)

def em_label_shift(P, prior_train, iters=200, tol=1e-7):
    """Saerens et al. (2002) EM: re-estimate the class shares of an unlabeled set and re-weight its probabilities accordingly.
    Uses only the model's test probabilities (no labels)."""
    pi = prior_train.copy()
    for _ in range(iters):
        Q = P * (pi / prior_train); Q /= Q.sum(1, keepdims=True)
        new = Q.mean(0)
        if np.abs(new - pi).max() < tol:
            break
        pi = new
    return Q, pi

def confusion_plot(y, pred, labels=None, title="Confusion matrix (row-normalised, out-of-fold)"):
    cm = confusion_matrix(y, pred, normalize="true")
    k = cm.shape[0]
    plt.figure(figsize=(6, 5)); plt.imshow(cm, cmap="Blues", vmin=0, vmax=1); plt.colorbar()
    for i in range(k):
        for j in range(k):
            plt.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center")
    plt.xticks(range(k), labels or range(k), rotation=30); plt.yticks(range(k), labels or range(k))
    plt.xlabel("predicted"); plt.ylabel("true"); plt.title(title); plt.tight_layout(); plt.show()

def save_outputs(task, pred, model_obj, extra=None, ids=None):
    id_col = ids if ids is not None else [f"ANS{i + 1:05d}" for i in range(len(pred))]
    df = pd.DataFrame({"id": id_col, "pred": pred})
    for k, v in (extra or {}).items():
        df[k] = v
    df.to_csv(OUT / f"task{task}_pred.csv", index=False)
    joblib.dump(model_obj, OUT / "models" / f"task{task}_model.joblib")
    print(f"saved {len(df)} predictions -> {OUT / f'task{task}_pred.csv'}")
    print(f"saved model -> {OUT / 'models' / f'task{task}_model.joblib'}")

## 2. Load Data

In [ ]:
train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")
TARGET = "class" if "class" in train.columns else "label"     # datacard calls it `class`
TRAIN_IMG, TEST_IMG = DATA / "train", DATA / "test"
if QUICK:
    train = pd.concat([d.sample(min(len(d), 20), random_state=SEED) for _, d in train.groupby(TARGET)], ignore_index=True)
    test = test.head(64).reset_index(drop=True)
print(f"Train shape: {train.shape}\nTest shape:  {test.shape}\nmissing labels: {int(train[TARGET].isna().sum())}")
train.head()

## 3. Exploratory Data Analysis (EDA)

EDA uses the **train** set only.

In [ ]:
vc = train[TARGET].value_counts(normalize=True).sort_index()
print(vc.set_axis(NAMES).round(3))
vc.set_axis(NAMES).plot(kind="bar", color="steelblue", figsize=(7, 4), title="Class share (train)")
plt.xticks(rotation=30); plt.show()

In [ ]:
fig, axes = plt.subplots(5, 5, figsize=(15, 7))
for k, row in enumerate(axes):
    sub = train.loc[train[TARGET] == k].sample(5, random_state=SEED)
    for ax, fn in zip(row, img_filenames(sub)):
        ax.imshow(Image.open(TRAIN_IMG / fn).convert("RGB")); ax.axis("off")
    row[0].set_title(NAMES[k], loc="left", fontsize=10)
plt.tight_layout(); plt.show()

In [ ]:
# image size / colour check on a random sample (train only)
sample = train.sample(min(300, len(train)), random_state=SEED)
sample_files = img_filenames(sample)
sizes = pd.Series([Image.open(TRAIN_IMG / fn).size for fn in sample_files]).value_counts()
print("image sizes (w, h):", dict(sizes))
mean_rgb = np.array([np.asarray(Image.open(TRAIN_IMG / fn).convert("RGB")).mean((0, 1)) for fn in sample_files])
print("mean RGB per class (sample):")
print(pd.DataFrame(mean_rgb, columns=list("RGB")).groupby(sample[TARGET].values).mean().set_axis(NAMES).round(1))

## 4. Data Preprocessing

- resize every screenshot to 128x256 (preserving the 230x120 landscape aspect ratio) and normalise with the ImageNet mean / std
- augmentation (train only): horizontal flip, colour jitter, small affine, random erasing - suited to game screenshots / HUD
- stratified 5-fold split (fold ids assigned on the **train** set); square-root class weights `balanced ** 0.5` computed per fold from the training part only

In [ ]:
skf = StratifiedKFold(CFG["n_folds"], shuffle=True, random_state=SEED)
train["fold"] = -1
for fold, (_, va) in enumerate(skf.split(train, train[TARGET])):
    train.loc[train.index[va], "fold"] = fold
print(train.groupby("fold")[TARGET].value_counts().unstack().set_axis(range(CFG["n_folds"])))

## 5. Model Training & Validation

### 5.1 Fine-tune ConvNeXt-Small with stratified K-fold
`convnext_small.fb_in22k_ft_in1k` (timm) fine-tuned end-to-end at 192x384 (env `IMG`) with strong augmentation (env `AUG=strong`: random-resized-crop 60-100 %, TrivialAugment, blur / grayscale, random erasing; `AUG=light` = previous setup): AdamW (`lr=2e-4`, `weight_decay=1e-2`) + 1-epoch warmup + cosine schedule, weight EMA (0.998), 12 epochs, label smoothing 0.05, mixed precision, square-root class-weighted cross-entropy (`balanced ** 0.5`).
The final EMA weights of each fold are used (no best-epoch selection, so OOF is unbiased); test probabilities use horizontal-flip TTA and are averaged over the 5 folds (env `FOLDS` selects folds).

**Why stronger augmentation:** train CV macro-F1 is ~0.99 but the leaderboard score is clearly lower. A train-vs-test classifier on 32x16 thumbnails separates the two sets with AUC 0.76, and the predicted test class shares differ strongly from train (7-11 Knights 39 % in train vs ~17 % predicted in test), so the gap is a **distribution shift**, not model capacity. Augmentations that vary framing, colour and sharpness make the model rely less on source-specific details. Finished folds are saved to `outputs/models/folds/` and reloaded on a re-run (resume after a Colab disconnect); OOF and test probabilities are saved as `.npy`.


In [ ]:
CKPT = OUT / "models" / "folds"; CKPT.mkdir(parents=True, exist_ok=True)

def run_fold(fold, model_name, extra=None):
    ck = CKPT / f"{model_name.split('.')[0]}_{CFG['aug']}_{'x'.join(map(str, CFG['img_size']))}_f{fold}{'_pl' if extra is not None else ''}.pt"
    if ck.exists() and not QUICK:   # resume after a Colab disconnect: finished folds are loaded, not retrained
        r = torch.load(ck, weights_only=False); print(f"loaded finished fold from {ck.name} (macro-F1 {r['f1']:.5f})")
        return r["val_idx"], r["pv"], r["pt"], r["state"]
    trn, val = train[train.fold != fold].reset_index(drop=True), train[train.fold == fold]
    val_idx, val = val.index.values, val.reset_index(drop=True)
    if extra is not None:   # pseudo-labelled test rows join the TRAINING part only; validation stays clean
        trn = pd.concat([trn.assign(file_name=img_filenames(trn)), extra], ignore_index=True)
    cw = compute_class_weight("balanced", classes=np.arange(CFG["num_classes"]), y=trn[TARGET].astype(int))
    w = torch.tensor(np.power(cw, 0.5), dtype=torch.float, device=CFG["device"])
    tr_loader = make_loader(trn, TRAIN_IMG, train_transform, True, TARGET)
    va_loader = make_loader(val, TRAIN_IMG, val_transform, False, TARGET)

    model = timm.create_model(model_name, pretrained=True, num_classes=CFG["num_classes"]).to(CFG["device"])
    crit = nn.CrossEntropyLoss(weight=w, label_smoothing=0.05)
    lr = MODEL_LR.get(model_name, 2e-4)
    head = [p for n, p in model.named_parameters() if n.startswith(("head", "classifier", "fc"))]
    ids = {id(p) for p in head}
    body = [p for p in model.parameters() if id(p) not in ids]
    opt = torch.optim.AdamW([{"params": body, "lr": lr}, {"params": head, "lr": lr * CFG["head_lr_mult"]}], weight_decay=CFG["weight_decay"])
    ema = timm.utils.ModelEmaV2(model, decay=CFG["ema_decay"])
    total, warm = CFG["epochs"] * len(tr_loader), CFG["warmup_epochs"] * len(tr_loader)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else
                                              max(1e-3, 0.5 * (1 + np.cos(np.pi * (s - warm) / max(1, total - warm)))))
    scaler = torch.amp.GradScaler("cuda", enabled=CFG["device"] == "cuda")

    best_f1, best_state, best_val = -1, None, None
    for ep in range(CFG["epochs"]):
        model.train(); loss_sum = 0.0; t0 = time.time()
        for x, y in tr_loader:
            x, y = x.to(CFG["device"]), y.to(CFG["device"])
            opt.zero_grad()
            with torch.autocast(CFG["device"], enabled=CFG["device"] == "cuda"):
                loss = crit(model(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step(); ema.update(model)
            loss_sum += loss.item() * len(x)
        pv = predict_proba(ema.module, va_loader, tta=False)
        f1 = f1_score(val[TARGET].astype(int), pv.argmax(1), average="macro")
        print(f"{model_name[:12]} fold {fold} | epoch {ep + 1:02d}/{CFG['epochs']} | train loss {loss_sum / (len(tr_loader) * CFG['batch_size']):.4f} | val macro-F1 {f1:.5f} | {time.time() - t0:4.0f}s")
    # keep the FINAL EMA weights (cosine schedule ends at ~0 LR) -> unbiased OOF, no best-epoch selection
    best_f1, best_val = f1, pv
    best_state = {k: v.detach().cpu().half() for k, v in ema.module.state_dict().items()}
    pt = predict_proba(ema.module, make_loader(test, TEST_IMG, val_transform, False), tta=True)
    print(f"fold {fold} final-EMA macro-F1: {best_f1:.5f}")
    if not QUICK:
        torch.save({"val_idx": val_idx, "pv": best_val, "pt": pt, "state": best_state, "f1": best_f1}, ck)
    return val_idx, best_val, pt, best_state

### 5.2 Run the folds and evaluate out-of-fold

In [ ]:
def run_all(extra=None, tag=""):
    """Train every backbone on every selected fold; return per-model OOF / test probabilities / states."""
    oof_m, pt_m, states = {}, {}, {}
    for mname in MODELS:
        oof = np.full((len(train), CFG["num_classes"]), np.nan)
        pt = np.zeros((len(test), CFG["num_classes"]))
        states[mname] = []
        for fold in CFG["train_folds"]:
            seed_everything(SEED + fold)
            vi, pv, pte, state = run_fold(fold, mname, extra)
            oof[vi] = pv
            pt += pte / len(CFG["train_folds"])
            states[mname].append(state)
        oof_m[mname], pt_m[mname] = oof, pt
        d = ~np.isnan(oof[:, 0])
        print()
        print(f">>> {tag}{mname}: OOF macro-F1 = {f1_score(y[d], oof[d].argmax(1), average='macro'):.5f}")
    return oof_m, pt_m, states

y = train[TARGET].astype(int).values
oof_m, pt_m, fold_states = run_all()

# ---- blend: plain mean of probabilities across backbones (nothing tuned) ----
oof = np.mean([oof_m[m] for m in MODELS], 0)
pt = np.mean([pt_m[m] for m in MODELS], 0)
done = ~np.isnan(oof[:, 0])
print()
print(f"BLEND OOF macro-F1 on {int(done.sum())} rows ({len(CFG['train_folds'])}/{CFG['n_folds']} folds): {f1_score(y[done], oof[done].argmax(1), average='macro'):.5f}")

# ---- optional pseudo-labelling (uses TEST images for training -> OFF by default, confirm with TA first) ----
if CFG["pseudo"] and not QUICK:
    keep = pt.max(1) >= CFG["pseudo_thr"]
    print(f"pseudo-label: {int(keep.sum())}/{len(test)} test images with conf >= {CFG['pseudo_thr']}")
    extra = pd.DataFrame({"file_name": ["../test/" + f for f in img_filenames(test)[keep]], TARGET: pt.argmax(1)[keep]})
    oof_m, pt_m, fold_states = run_all(extra, tag="[pseudo] ")
    oof = np.mean([oof_m[m] for m in MODELS], 0)   # OOF stays honest: validation folds never contain pseudo rows
    pt = np.mean([pt_m[m] for m in MODELS], 0)
    print(f"BLEND OOF after pseudo-labelling: {f1_score(y[done], oof[done].argmax(1), average='macro'):.5f}")

# ---- stacking: multinomial logistic regression on log-probabilities of all backbones (fit on train OOF only) ----
if done.all() and len(MODELS) > 1:
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import cross_val_predict
    feat = lambda d: np.hstack([np.log(np.clip(d[m], 1e-6, 1)) for m in MODELS])
    Xs, Xt = feat(oof_m), feat(pt_m)
    meta = LogisticRegression(C=1.0, max_iter=2000, class_weight="balanced")
    cv = StratifiedKFold(5, shuffle=True, random_state=SEED + 99)
    sp = cross_val_predict(meta, Xs, y, cv=cv, method="predict_proba")   # nested CV -> honest stacked score
    f_blend, f_stack = f1_score(y, oof.argmax(1), average="macro"), f1_score(y, sp.argmax(1), average="macro")
    print(f"mean-blend OOF {f_blend:.5f} | stacked (nested CV) {f_stack:.5f}")
    if f_stack > f_blend:
        meta.fit(Xs, y); pt = meta.predict_proba(Xt); oof = sp
        print("-> using the stacked model for test predictions")
    else:
        print("-> stacking does not beat the mean blend; keeping the mean blend")
else:
    print("stacking skipped (needs all folds and >= 2 backbones)")

np.save(OUT / "task4_oof_proba.npy", oof); np.save(OUT / "task4_test_proba.npy", pt)   # keep probabilities -> post-processing without retraining

# ---- optional label-shift correction (uses unlabeled TEST predictions -> OFF by default, confirm with TA first) ----
prior_tr = np.bincount(y, minlength=CFG["num_classes"]) / len(y)
if CFG["prior_em"]:
    pt_em, pi_test = em_label_shift(pt, prior_tr)
    print("estimated test class shares (EM):", dict(zip(NAMES, pi_test.round(3))), "| train:", dict(zip(NAMES, prior_tr.round(3))))
    print(f"labels changed by the correction: {(pt_em.argmax(1) != pt.argmax(1)).sum()} / {len(pt)}")
    pt = pt_em

print()
print(classification_report(y[done], oof[done].argmax(1), labels=range(CFG["num_classes"]), target_names=NAMES, digits=4, zero_division=0))
confusion_plot(y[done], oof[done].argmax(1), NAMES)

Reference run on Colab T4 (`convnext_small.fb_in22k_ft_in1k`, 128x256, fold 0, 8 epochs): best-epoch macro-F1 = 0.99010 (epoch 7/8); predicted test class shares = 16.28% / 23.43% / 15.77% / 26.89% / 17.63%.

> Previous single-fold reference (128x256, fold 0, 8 epochs, best-epoch) = 0.9901; the new 5-fold / EMA / 192x384 setup should be compared against the printed OOF macro-F1.

## 6. Predict on the Test Set & Save

In [ ]:
pred = pt.argmax(1)
share = pd.DataFrame({
    "train": pd.Series(y).value_counts(normalize=True).reindex(range(CFG["num_classes"]), fill_value=0),
    "test predicted": pd.Series(pred).value_counts(normalize=True).reindex(range(CFG["num_classes"]), fill_value=0),
}).set_axis(NAMES)
print(share.round(3))
share.plot(kind="bar", figsize=(7, 4), color=["steelblue", "tomato"], title="Class share: train vs predicted test")
plt.xticks(rotation=30); plt.show()
save_outputs(TASK, pred, {"name": MODEL_NAME, "models": MODELS, "fold_state_dicts": fold_states, "cfg": CFG, "classes": NAMES},
             {f"p{k}": pt[:, k] for k in range(CFG["num_classes"])}, ids=test["id"].values)